# 5. Ejecutar Pipeline Gold

Propósito: Construir y guardar los marts gold desde silver parquet.

In [1]:
# --- Bootstrap del entorno (Windows + VSCode) ---
# VSCode inyecta el .env del repo (con rutas Linux para JAVA_HOME/HADOOP_HOME)
# dentro del kernel; en Windows esas rutas no existen y Spark no arranca.
# Ademas los notebooks corren desde notebooks/, por lo que fijamos el cwd y el
# sys.path en la raiz del repo para que 'data/...' e 'import app' funcionen.
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if not os.path.isdir(os.environ.get("JAVA_HOME", "")):
    os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot"
_hadoop = os.environ.get("HADOOP_HOME", "")
if not (os.path.isabs(_hadoop) and os.path.isdir(_hadoop)):
    os.environ["HADOOP_HOME"] = str(ROOT / "lib" / "hadoop")

print("ROOT:", ROOT)
print("JAVA_HOME:", os.environ["JAVA_HOME"])

ROOT: d:\Universidad\CICLO_VII\BigData\Proyecto\EP-GDM-G6
JAVA_HOME: C:\Program Files\Java\jdk-21


In [2]:
from app.utils.spark import SparkClient
spark_client = SparkClient()
spark = spark_client.get_session()

In [3]:
from app.gold import transforms
from pathlib import Path
silver_dir = Path("data/silver")
dims, marts = transforms.build_all(spark, silver_dir)
print(f"Dimensiones gold: {len(dims)}, Marts: {len(marts)}")

2026-06-18 21:59:03,283 - INFO - Leyendo silver parquets
2026-06-18 21:59:09,098 - INFO - Leyendo fact tables de silver
2026-06-18 21:59:15,781 - INFO - Construidas 3 dims y 5 marts gold


Dimensiones gold: 3, Marts: 5


In [4]:
from app.gold.parquet_loader import save_all_to_parquet
save_all_to_parquet(dims, marts, Path("data/gold"))
print("Gold parquet guardados en data/gold/")

2026-06-18 21:59:21,093 - INFO - Saved DIM_CALENDARIO to data\gold\DIM_CALENDARIO.parquet
2026-06-18 21:59:21,937 - INFO - Saved DIM_ANIO to data\gold\DIM_ANIO.parquet
2026-06-18 21:59:22,727 - INFO - Saved DIM_GEOGRAFIA to data\gold\DIM_GEOGRAFIA.parquet
2026-06-18 21:59:36,910 - INFO - Saved MART_INGRESOS_GEOGRAFICO to data\gold\MART_INGRESOS_GEOGRAFICO.parquet
2026-06-18 21:59:40,698 - INFO - Saved MART_INGRESOS_CLASIFICADOR to data\gold\MART_INGRESOS_CLASIFICADOR.parquet
2026-06-18 21:59:43,839 - INFO - Saved MART_INGRESOS_EJECUTORA to data\gold\MART_INGRESOS_EJECUTORA.parquet
2026-06-18 21:59:46,931 - INFO - Saved MART_PREDIAL to data\gold\MART_PREDIAL.parquet
2026-06-18 22:00:28,480 - INFO - Saved MART_RENAMU to data\gold\MART_RENAMU.parquet


Gold parquet guardados en data/gold/


In [5]:
import os
for f in sorted(os.listdir("data/gold")):
    if f.endswith(".parquet"):
        df = spark.read.parquet(f"data/gold/{f}")
        print(f"{f}: {df.count():,} filas")

DIM_ANIO.parquet: 18 filas
DIM_CALENDARIO.parquet: 312 filas
DIM_GEOGRAFIA.parquet: 1,891 filas
MART_INGRESOS_CLASIFICADOR.parquet: 1,503 filas
MART_INGRESOS_EJECUTORA.parquet: 7,074 filas
MART_INGRESOS_GEOGRAFICO.parquet: 84,793 filas
MART_PREDIAL.parquet: 230,942 filas
MART_RENAMU.parquet: 12,770,291 filas
